# Práctica 3 - Transporte RD (Green AI)

Notebook independiente para **Práctica 3**, construida en paralelo a la notebook de **Práctica 2**, con el propósito de comparar ambas aproximaciones.

## Propósito
Analizar el algoritmo de clasificación utilizado en la práctica 2 no solo desde el rendimiento predictivo, sino también desde la perspectiva de **algoritmos verdes (Green AI)**, evaluando:

- variación del tamaño de la muestra,
- variación del número de variables,
- tiempo de entrenamiento,
- estrategias para reducir el coste computacional.

## Alcance metodológico
Esta notebook:
1. Reutiliza el pipeline general del proyecto.
2. Ejecuta experimentos controlados de eficiencia computacional.
3. Compara configuraciones base y optimizadas.
4. Genera tablas y gráficos para el informe.
5. Incorpora una sección explícita de transparencia en el uso de IA.


In [ ]:
# ============================================================
# OPCIONAL - Clonar repositorio en Colab si aún no existe
# Descomenta estas líneas si vas a trabajar directamente desde GitHub.
# ============================================================

 !git clone https://github.com/edjnolasco/transport-ml-rd.git /content/transport-ml-rd


In [ ]:
# ============================================================
# Configuración general del entorno
# ============================================================

import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = Path("/content/transport-ml-rd").resolve()
sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Existe el proyecto:", PROJECT_ROOT.exists())


In [ ]:
# ============================================================
# Importaciones del proyecto
# Ajusta este bloque solo si algún módulo cambió de nombre en tu repo.
# ============================================================

from src.config import build_config
from src.data_processing import load_dataset, basic_cleaning
from src.features import split_features_target

config = build_config(PROJECT_ROOT)

print("Archivo de entrada esperado por config:", config.input_file)
print("Archivo procesado esperado por config:", config.processed_file)


## Carga y limpieza de datos

Este bloque reutiliza la lógica central de la práctica 2, pero deja los experimentos Green AI en una notebook separada.


In [ ]:
# ============================================================
# Carga y limpieza de datos
# ============================================================

df_raw = load_dataset(config.input_file)
print("Dataset original:", df_raw.shape)

df_clean = basic_cleaning(df_raw.copy())
print("Dataset limpio:", df_clean.shape)

display(df_clean.head())


In [ ]:
# ============================================================
# Separación de variables predictoras y variable objetivo
# Si tu función split_features_target requiere otra firma, ajústala aquí.
# ============================================================

X, y = split_features_target(df_clean)

print("Forma de X:", X.shape)
print("Longitud de y:", len(y))
print("Distribución de clases:")
display(pd.Series(y).value_counts(dropna=False))


In [ ]:
# ============================================================
# Conversión segura para experimentos Green AI
# Esta transformación no altera tu pipeline base original.
# ============================================================

X_green = X.copy()
X_green = pd.get_dummies(X_green, drop_first=True)
X_green = X_green.fillna(0)
X_green.columns = [str(col) for col in X_green.columns]

print("Forma de X_green:", X_green.shape)
print("Primeras columnas:")
display(pd.Series(X_green.columns[:15]))


## Función central de evaluación

La siguiente función mide el tiempo de entrenamiento y calcula métricas predictivas para cada configuración experimental.


In [ ]:
# ============================================================
# Función base de entrenamiento y evaluación
# ============================================================

def train_and_evaluate_model(
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    experiment_name,
    sample_fraction,
    n_features,
    extra_info=None,
):
    local_model = clone(model)

    start_time = time.perf_counter()
    local_model.fit(X_train, y_train)
    end_time = time.perf_counter()

    train_time = end_time - start_time
    y_pred = local_model.predict(X_test)

    result = {
        "experiment": experiment_name,
        "model": local_model.__class__.__name__,
        "sample_fraction": sample_fraction,
        "n_features": n_features,
        "train_time_sec": train_time,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_weighted": precision_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
    }

    try:
        if hasattr(local_model, "predict_proba"):
            y_proba = local_model.predict_proba(X_test)
            if y_proba.shape[1] == 2:
                result["roc_auc"] = roc_auc_score(y_test, y_proba[:, 1])
            else:
                result["roc_auc"] = roc_auc_score(
                    y_test, y_proba, multi_class="ovr", average="weighted"
                )
        else:
            result["roc_auc"] = np.nan
    except Exception:
        result["roc_auc"] = np.nan

    if extra_info:
        result.update(extra_info)

    return result


In [ ]:
# ============================================================
# Modelo base de referencia
# Se usa LogisticRegression como baseline interpretable y eficiente.
# ============================================================

base_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)

print(base_model)


In [ ]:
# ============================================================
# Split base común
# ============================================================

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_green,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


## Experimento 0 - Línea base

Esta configuración representa el punto de partida para comparar los demás experimentos de Green AI.


In [ ]:
# ============================================================
# Experimento 0 - Línea base
# ============================================================

green_results = []

baseline_result = train_and_evaluate_model(
    model=base_model,
    X_train=X_train_base,
    X_test=X_test_base,
    y_train=y_train_base,
    y_test=y_test_base,
    experiment_name="baseline_full_data_full_features",
    sample_fraction=1.0,
    n_features=X_train_base.shape[1],
    extra_info={"strategy": "baseline"},
)

green_results.append(baseline_result)

display(pd.DataFrame([baseline_result]))


## Experimento 1 - Variación del tamaño de muestra

Se analiza cómo cambia el tiempo de entrenamiento y el rendimiento cuando se modifica la fracción de datos utilizada.


In [ ]:
# ============================================================
# Experimento 1 - Variación del tamaño de muestra
# ============================================================

sample_fractions = [0.4, 0.6, 0.8, 1.0]

for frac in sample_fractions:
    if frac < 1.0:
        X_sub, _, y_sub, _ = train_test_split(
            X_green,
            y,
            train_size=frac,
            stratify=y,
            random_state=RANDOM_STATE,
        )
    else:
        X_sub, y_sub = X_green.copy(), pd.Series(y).copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X_sub,
        y_sub,
        test_size=0.2,
        stratify=y_sub,
        random_state=RANDOM_STATE,
    )

    result = train_and_evaluate_model(
        model=base_model,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        experiment_name="sample_size_variation",
        sample_fraction=frac,
        n_features=X_train.shape[1],
        extra_info={"strategy": "vary_sample_size"},
    )

    green_results.append(result)

df_sample_results = pd.DataFrame(
    [r for r in green_results if r["experiment"] == "sample_size_variation"]
).sort_values("sample_fraction")

display(df_sample_results)


## Experimento 2 - Variación del número de variables

Se aplica selección de variables para estudiar el efecto de la dimensionalidad sobre el coste computacional y el rendimiento.


In [ ]:
# ============================================================
# Experimento 2 - Variación del número de variables
# ============================================================

total_features = X_green.shape[1]

candidate_feature_sizes = sorted(
    list(
        {
            min(3, total_features),
            min(5, total_features),
            min(7, total_features),
            max(1, int(total_features * 0.5)),
            total_features,
        }
    )
)
candidate_feature_sizes = [k for k in candidate_feature_sizes if 1 <= k <= total_features]

feature_selection_results = []

for k in candidate_feature_sizes:
    selector = SelectKBest(score_func=f_classif, k=k)
    X_selected = selector.fit_transform(X_green, y)

    X_train, X_test, y_train, y_test = train_test_split(
        X_selected,
        y,
        test_size=0.2,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    result = train_and_evaluate_model(
        model=base_model,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        experiment_name="feature_count_variation",
        sample_fraction=1.0,
        n_features=k,
        extra_info={"strategy": "select_k_best"},
    )

    green_results.append(result)
    feature_selection_results.append(result)

df_feature_results = pd.DataFrame(feature_selection_results).sort_values("n_features")

display(df_feature_results)


## Experimento 3 - Regularización

Se evalúa el efecto del parámetro `C` en Logistic Regression como mecanismo de control de complejidad.


In [ ]:
# ============================================================
# Experimento 3 - Regularización
# ============================================================

C_values = [0.01, 0.1, 1.0, 10.0]
regularization_results = []

for c_value in C_values:
    regularized_model = LogisticRegression(
        C=c_value,
        max_iter=1000,
        random_state=RANDOM_STATE,
    )

    result = train_and_evaluate_model(
        model=regularized_model,
        X_train=X_train_base,
        X_test=X_test_base,
        y_train=y_train_base,
        y_test=y_test_base,
        experiment_name="regularization_variation",
        sample_fraction=1.0,
        n_features=X_train_base.shape[1],
        extra_info={
            "strategy": "regularization",
            "C_value": c_value,
        },
    )

    green_results.append(result)
    regularization_results.append(result)

df_regularization_results = pd.DataFrame(regularization_results).sort_values("C_value")

display(df_regularization_results)


## Experimento 4 - Sustitución por un modelo más simple

Se compara el baseline con un modelo de árbol poco profundo para evaluar si una estructura más simple puede ofrecer una relación favorable entre rendimiento y coste.


In [ ]:
# ============================================================
# Experimento 4 - Modelo alternativo más simple
# ============================================================

simple_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=RANDOM_STATE,
)

simple_model_result = train_and_evaluate_model(
    model=simple_model,
    X_train=X_train_base,
    X_test=X_test_base,
    y_train=y_train_base,
    y_test=y_test_base,
    experiment_name="simple_model_comparison",
    sample_fraction=1.0,
    n_features=X_train_base.shape[1],
    extra_info={"strategy": "simpler_model_max_depth_3"},
)

green_results.append(simple_model_result)

display(pd.DataFrame([simple_model_result]))


## Experimento 5 - Poda en modelos basados en árboles

Se modifica la profundidad máxima del árbol para analizar el efecto de la complejidad estructural sobre el coste de entrenamiento y el desempeño.


In [ ]:
# ============================================================
# Experimento 5 - Poda / complejidad del árbol
# ============================================================

tree_depths = [2, 3, 5, None]
tree_pruning_results = []

for depth in tree_depths:
    tree_model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=RANDOM_STATE,
    )

    result = train_and_evaluate_model(
        model=tree_model,
        X_train=X_train_base,
        X_test=X_test_base,
        y_train=y_train_base,
        y_test=y_test_base,
        experiment_name="tree_pruning_variation",
        sample_fraction=1.0,
        n_features=X_train_base.shape[1],
        extra_info={
            "strategy": "tree_pruning",
            "max_depth": depth if depth is not None else "None",
        },
    )

    green_results.append(result)
    tree_pruning_results.append(result)

df_tree_results = pd.DataFrame(tree_pruning_results)

display(df_tree_results)


## Consolidación general de resultados


In [ ]:
# ============================================================
# Consolidación general de resultados
# ============================================================

df_green_results = pd.DataFrame(green_results)

preferred_columns = [
    "experiment",
    "strategy",
    "model",
    "sample_fraction",
    "n_features",
    "train_time_sec",
    "accuracy",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
    "roc_auc",
    "C_value",
    "max_depth",
]
existing_columns = [col for col in preferred_columns if col in df_green_results.columns]
remaining_columns = [col for col in df_green_results.columns if col not in existing_columns]
df_green_results = df_green_results[existing_columns + remaining_columns]

print("Resultados consolidados:")
display(df_green_results)


In [ ]:
# ============================================================
# Resumen ordenado por tiempo de entrenamiento
# ============================================================

df_green_summary = df_green_results.sort_values(
    by=["train_time_sec", "f1_weighted"],
    ascending=[True, False],
).reset_index(drop=True)

display(df_green_summary)


## Visualizaciones


In [ ]:
# ============================================================
# Gráfico 1 - Tiempo de entrenamiento vs tamaño de muestra
# ============================================================

if not df_sample_results.empty:
    plt.figure(figsize=(8, 5))
    plt.plot(
        df_sample_results["sample_fraction"],
        df_sample_results["train_time_sec"],
        marker="o",
    )
    plt.title("Tiempo de entrenamiento vs tamaño de muestra")
    plt.xlabel("Fracción de muestra")
    plt.ylabel("Tiempo de entrenamiento (segundos)")
    plt.grid(True)
    plt.show()


In [ ]:
# ============================================================
# Gráfico 2 - F1 ponderado vs tamaño de muestra
# ============================================================

if not df_sample_results.empty:
    plt.figure(figsize=(8, 5))
    plt.plot(
        df_sample_results["sample_fraction"],
        df_sample_results["f1_weighted"],
        marker="o",
    )
    plt.title("F1 ponderado vs tamaño de muestra")
    plt.xlabel("Fracción de muestra")
    plt.ylabel("F1 ponderado")
    plt.grid(True)
    plt.show()


In [ ]:
# ============================================================
# Gráfico 3 - Tiempo de entrenamiento vs número de variables
# ============================================================

if not df_feature_results.empty:
    plt.figure(figsize=(8, 5))
    plt.plot(
        df_feature_results["n_features"],
        df_feature_results["train_time_sec"],
        marker="o",
    )
    plt.title("Tiempo de entrenamiento vs número de variables")
    plt.xlabel("Número de variables")
    plt.ylabel("Tiempo de entrenamiento (segundos)")
    plt.grid(True)
    plt.show()


In [ ]:
# ============================================================
# Gráfico 4 - F1 ponderado vs número de variables
# ============================================================

if not df_feature_results.empty:
    plt.figure(figsize=(8, 5))
    plt.plot(
        df_feature_results["n_features"],
        df_feature_results["f1_weighted"],
        marker="o",
    )
    plt.title("F1 ponderado vs número de variables")
    plt.xlabel("Número de variables")
    plt.ylabel("F1 ponderado")
    plt.grid(True)
    plt.show()


In [ ]:
# ============================================================
# Gráfico 5 - Trade-off entre rendimiento y coste computacional
# ============================================================

plt.figure(figsize=(10, 6))
plt.scatter(
    df_green_results["train_time_sec"],
    df_green_results["f1_weighted"],
)

for _, row in df_green_results.iterrows():
    label = f"{row['model']} | {row['strategy']}"
    plt.annotate(
        label,
        (row["train_time_sec"], row["f1_weighted"]),
        fontsize=8,
        alpha=0.8,
    )

plt.title("Trade-off entre F1 ponderado y tiempo de entrenamiento")
plt.xlabel("Tiempo de entrenamiento (segundos)")
plt.ylabel("F1 ponderado")
plt.grid(True)
plt.show()


## Selección de configuración Green AI recomendada

Criterio de trabajo: entre las configuraciones cuyo F1 sea al menos el 95% del mejor F1 observado, seleccionar la de menor tiempo de entrenamiento.


In [ ]:
# ============================================================
# Selección de mejor configuración Green AI
# ============================================================

best_f1 = df_green_results["f1_weighted"].max()
threshold = best_f1 * 0.95

df_candidate_green = df_green_results[
    df_green_results["f1_weighted"] >= threshold
].copy()

df_candidate_green = df_candidate_green.sort_values("train_time_sec").reset_index(drop=True)

print("Mejor F1 global:", best_f1)
print("Umbral Green AI (95% del mejor F1):", threshold)

display(df_candidate_green)

if not df_candidate_green.empty:
    recommended_green_config = df_candidate_green.iloc[0]
    print("Configuración Green AI recomendada:")
    display(pd.DataFrame([recommended_green_config]))


In [ ]:
# ============================================================
# Comparación final entre baseline, mejor Green AI y modelo simple
# ============================================================

comparison_rows = [baseline_result]

if "recommended_green_config" in globals():
    comparison_rows.append(recommended_green_config.to_dict())

comparison_rows.append(simple_model_result)

df_final_comparison = pd.DataFrame(comparison_rows).drop_duplicates()

display(df_final_comparison)


In [ ]:
# ============================================================
# Guardado de resultados
# ============================================================

reports_dir = PROJECT_ROOT / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

green_results_path = reports_dir / "green_ai_results.csv"
green_summary_path = reports_dir / "green_ai_summary.csv"
green_final_comparison_path = reports_dir / "green_ai_final_comparison.csv"

df_green_results.to_csv(green_results_path, index=False)
df_green_summary.to_csv(green_summary_path, index=False)
df_final_comparison.to_csv(green_final_comparison_path, index=False)

print("Archivos guardados:")
print(green_results_path)
print(green_summary_path)
print(green_final_comparison_path)


In [ ]:
# ============================================================
# Síntesis descriptiva inicial
# ============================================================

print("Resumen descriptivo inicial")
print("-" * 40)
print(f"Total de experimentos ejecutados: {len(df_green_results)}")
print(f"Mejor F1 ponderado observado: {df_green_results['f1_weighted'].max():.4f}")
print(f"Menor tiempo de entrenamiento observado: {df_green_results['train_time_sec'].min():.6f} segundos")
print(f"Mayor tiempo de entrenamiento observado: {df_green_results['train_time_sec'].max():.6f} segundos")

fastest_row = df_green_results.sort_values("train_time_sec").iloc[0]
best_f1_row = df_green_results.sort_values("f1_weighted", ascending=False).iloc[0]

print("\nConfiguración más rápida:")
display(pd.DataFrame([fastest_row]))

print("Configuración con mejor F1:")
display(pd.DataFrame([best_f1_row]))


## Transparencia en el uso de IA

Este notebook fue desarrollado con apoyo de herramientas de inteligencia artificial en tareas de estructuración metodológica, organización del flujo experimental y generación de código base.

La inteligencia artificial no fue utilizada como fuente de resultados, evidencia empírica ni mecanismo de validación experimental. La ejecución de los experimentos, la revisión del comportamiento del modelo, la interpretación de los resultados y las decisiones metodológicas finales fueron realizadas bajo supervisión y control humano.

En consecuencia, la IA se utilizó exclusivamente como herramienta de apoyo, manteniéndose la responsabilidad analítica y científica en la persona autora del trabajo.
